# Notebook 4: Soft-Margin SVM

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours | **Week 10 — Support Vector Machines & Kernel Methods**

---

## Why This Matters

Real-world data is messy. The hard-margin SVM from Notebook 2 requires perfect linear separability — a single misplaced point makes the entire problem infeasible. Soft-margin SVM is the practical version that:
- Handles overlapping classes gracefully
- Introduces a tunable trade-off between margin width and training errors
- Connects directly to **hinge loss** and **regularization** (bridging to linear models)
- Is what sklearn's `SVC` actually implements by default

---

## Real-World Analogy First: The Spam Filter That Allows Mistakes

Imagine you're building a spam filter, but your training data is **messy**:
- Some legitimate emails from your bank use phrases like "Limited time offer" (looks spammy)
- Some spam emails are professionally written with low spam-word density (looks legitimate)

**Hard rule (hard-margin):** "Every single email MUST be on the correct side of the boundary." Result: the boundary bends awkwardly to accommodate outliers, becoming fragile and overfitted.

**Soft rule (soft-margin):** "Mostly correct is fine. But I'll charge you a **penalty fee** for each mistake." The fee size is controlled by **C**:
- **High C = strict judge:** Even one mistake is very costly → boundary bends to avoid mistakes → narrow margin → may overfit
- **Low C = lenient judge:** Mistakes are cheap → boundary ignores outliers → wide margin → may underfit

The optimal C balances the width of the margin against the cost of training errors.

---

## Context: Email Spam Classification with Overlap

We use the same two features as Notebook 3:
- **word_count**: total number of words in the email  
- **spam_word_ratio**: fraction of words that are common spam words

But now the classes **overlap** — about 15% of emails are in the ambiguous region between spam and ham.

## The Mathematics of Soft-Margin SVM

### Slack Variables: Measuring How Bad a Violation Is

For each training point $(x_i, y_i)$, introduce a slack variable $\xi_i \geq 0$:

$$y_i(w \cdot x_i + b) \geq 1 - \xi_i$$

The slack $\xi_i$ measures the degree of violation:

| Condition | $\xi_i$ value | Interpretation |
|---|---|---|
| $y_i f(x_i) \geq 1$ | $\xi_i = 0$ | Correct, outside or on margin |
| $0 < y_i f(x_i) < 1$ | $0 < \xi_i < 1$ | Correct but inside margin |
| $y_i f(x_i) = 0$ | $\xi_i = 1$ | On the decision boundary (wrong side) |
| $y_i f(x_i) < 0$ | $\xi_i > 1$ | **Misclassified** |

### The Soft-Margin Objective

$$\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C \sum_{i=1}^{n} \xi_i$$

subject to: $y_i(w \cdot x_i + b) \geq 1 - \xi_i$ and $\xi_i \geq 0$ for all $i$

- **First term** $\frac{1}{2}\|w\|^2$: maximize the margin (geometric term)
- **Second term** $C \sum \xi_i$: penalize margin violations
- **C**: the trade-off parameter (hyperparameter to tune)

### Connection to Hinge Loss

At optimality, $\xi_i = \max(0, 1 - y_i f(x_i))$ — this is exactly the **hinge loss**!

So the objective becomes:
$$\frac{1}{2}\|w\|^2 + C \sum_{i} \max\left(0, 1 - y_i f(x_i)\right)$$

This is **L2-regularized hinge loss minimization** — SVM is a special case of regularized ERM.

### Dual of Soft-Margin SVM

Same dual as hard-margin, but with a **box constraint**:

$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j (x_i \cdot x_j)$$

subject to: $0 \leq \alpha_i \leq C$ and $\sum_i \alpha_i y_i = 0$

The only change: $\alpha_i$ is now bounded above by $C$ instead of being unbounded.

### Support Vector Types in Soft-Margin

| $\alpha_i$ value | $\xi_i$ | Type |
|---|---|---|
| $\alpha_i = 0$ | $\xi_i = 0$ | Non-SV (correctly classified, outside margin) |
| $0 < \alpha_i < C$ | $\xi_i = 0$ | **Margin SV** (exactly on the margin) |
| $\alpha_i = C$ | $\xi_i > 0$ | **Violator SV** (inside margin or misclassified) |

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)
sns.set_theme(style='whitegrid')

print("All libraries loaded successfully.")
print(f"NumPy: {np.__version__}")

In [ ]:
# ─── Section 1: Generate Overlapping 2D Spam Dataset ─────────────────────────
np.random.seed(42)

N_TOTAL = 200
n_spam  = 100
n_ham   = 100

# Spam emails: moderate–high word count, higher spam ratio
# Using a wider standard deviation to create overlap
spam_wc   = np.random.normal(loc=280, scale=70, size=n_spam)   # word_count
spam_swr  = np.random.normal(loc=0.65, scale=0.12, size=n_spam) # spam_word_ratio

# Ham emails: lower word count, lower spam ratio
ham_wc    = np.random.normal(loc=160, scale=70, size=n_ham)
ham_swr   = np.random.normal(loc=0.25, scale=0.12, size=n_ham)

# Clip to realistic ranges
spam_wc  = np.clip(spam_wc,  30, 600)
spam_swr = np.clip(spam_swr, 0.01, 0.99)
ham_wc   = np.clip(ham_wc,   30, 600)
ham_swr  = np.clip(ham_swr,  0.01, 0.99)

X_raw = np.vstack([
    np.column_stack([spam_wc, spam_swr]),
    np.column_stack([ham_wc,  ham_swr])
])
y = np.hstack([np.ones(n_spam), -np.ones(n_ham)])

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Identify the "overlap" region — points where the classes mix
# Simple heuristic: points within 1 std of the midpoint on first feature
midpoint_x0 = 0.0  # after scaling
near_boundary = np.abs(X[:, 0]) < 0.8
n_overlap = near_boundary.sum()

df = pd.DataFrame({
    'word_count_raw':      X_raw[:, 0],
    'spam_word_ratio_raw': X_raw[:, 1],
    'word_count_scaled':   X[:, 0],
    'spam_ratio_scaled':   X[:, 1],
    'label':               y,
    'class':               ['Spam' if yi == 1 else 'Ham' for yi in y]
})

print(f"Dataset shape:        {X.shape}")
print(f"Spam samples:         {n_spam}")
print(f"Ham samples:          {n_ham}")
print(f"Approx. overlap zone: {n_overlap} points near decision boundary")
print(f"\nFeature stats after scaling:")
print(df[['word_count_scaled', 'spam_ratio_scaled']].describe().round(3))

In [ ]:
# ─── Visualize the Overlapping Dataset ────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Spam': '#e74c3c', 'Ham': '#2ecc71'}

# Raw features
for cls, grp in df.groupby('class'):
    axes[0].scatter(grp['word_count_raw'], grp['spam_word_ratio_raw'],
                    c=colors[cls], label=cls, alpha=0.65, edgecolors='k', linewidths=0.3, s=55)
# Shade the overlap region
axes[0].axvspan(180, 280, alpha=0.08, color='purple', label='Overlap zone')
axes[0].set_xlabel('Word Count', fontsize=12)
axes[0].set_ylabel('Spam Word Ratio', fontsize=12)
axes[0].set_title('Raw Features — Classes Overlap!', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Scaled features
for cls, grp in df.groupby('class'):
    axes[1].scatter(grp['word_count_scaled'], grp['spam_ratio_scaled'],
                    c=colors[cls], label=cls, alpha=0.65, edgecolors='k', linewidths=0.3, s=55)
axes[1].axvspan(-0.8, 0.8, alpha=0.08, color='purple', label='Overlap zone')
axes[1].set_xlabel('Word Count (Scaled)', fontsize=12)
axes[1].set_ylabel('Spam Word Ratio (Scaled)', fontsize=12)
axes[1].set_title('Standardized Features — Still Overlapping', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Email Dataset: NOT Linearly Separable (Overlap ~15%)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("This dataset CANNOT be perfectly separated by a straight line.")
print("A hard-margin SVM will either fail or produce a degenerate boundary.")

In [ ]:
# ─── Section 2: Show Failure of Hard-Margin SVM ───────────────────────────────

print("=" * 60)
print("HARD-MARGIN SVM: What happens with overlapping data?")
print("=" * 60)

# sklearn will still return a solution for C=1e6 (numerical approximation),
# but the margin will be essentially zero and/or many SVs will be forced
svc_hard = SVC(kernel='linear', C=1e6, max_iter=10000)
svc_hard.fit(X, y)
w_hard = svc_hard.coef_[0]
b_hard = svc_hard.intercept_[0]

pred_hard    = svc_hard.predict(X)
acc_hard     = accuracy_score(y, pred_hard)
n_sv_hard    = svc_hard.n_support_.sum()
margin_hard  = 2 / np.linalg.norm(w_hard)

print(f"\nC = 1e6 (approaching hard-margin):")
print(f"  Training accuracy:    {acc_hard * 100:.1f}%")
print(f"  Number of SVs:        {n_sv_hard} / {len(y)} ({n_sv_hard/len(y)*100:.1f}%)")
print(f"  Margin width (2/||w||): {margin_hard:.6f}  ← VERY narrow!")
print(f"  ||w||:                {np.linalg.norm(w_hard):.4f}  ← large = narrow margin")

# Now compare with a reasonable C
svc_medium = SVC(kernel='linear', C=1.0)
svc_medium.fit(X, y)
w_med   = svc_medium.coef_[0]
b_med   = svc_medium.intercept_[0]
acc_med = accuracy_score(y, svc_medium.predict(X))
n_sv_med = svc_medium.n_support_.sum()
margin_med = 2 / np.linalg.norm(w_med)

print(f"\nC = 1 (soft-margin, balanced):")
print(f"  Training accuracy:    {acc_med * 100:.1f}%")
print(f"  Number of SVs:        {n_sv_med} / {len(y)} ({n_sv_med/len(y)*100:.1f}%)")
print(f"  Margin width (2/||w||): {margin_med:.6f}  ← much wider!")
print(f"  ||w||:                {np.linalg.norm(w_med):.4f}")

print(f"\n→ Hard-margin: {n_sv_hard} SVs, margin = {margin_hard:.4f}")
print(f"→ Soft-margin: {n_sv_med} SVs, margin = {margin_med:.4f}  (traded accuracy for width)")

In [ ]:
# ─── Visualize Hard-margin vs Soft-margin Boundaries ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def plot_svm_boundary(ax, X, y, svc, title, colors_dict):
    """Plot decision boundary and margin for a fitted SVC."""
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
    )
    Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-10, 0, 10], colors=['#aed6f1', '#fadbd8'], alpha=0.35)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               colors=['#2980b9', '#2c3e50', '#c0392b'],
               linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])

    is_spam = (y == 1)
    ax.scatter(X[is_spam, 0],  X[is_spam, 1],  c='#e74c3c', s=45, alpha=0.65, edgecolors='none', label='Spam')
    ax.scatter(X[~is_spam, 0], X[~is_spam, 1], c='#2ecc71', s=45, alpha=0.65, edgecolors='none', label='Ham')

    sv_idx = svc.support_
    ax.scatter(X[sv_idx, 0], X[sv_idx, 1], s=200, facecolors='none',
               edgecolors='k', linewidths=2, label=f'SVs ({len(sv_idx)})')

    w   = svc.coef_[0]
    margin_w = 2 / np.linalg.norm(w)
    ax.set_title(f'{title}\n(SVs={len(sv_idx)}, margin={margin_w:.3f})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Word Count (Scaled)', fontsize=11)
    ax.set_ylabel('Spam Word Ratio (Scaled)', fontsize=11)
    ax.legend(fontsize=9)


plot_svm_boundary(axes[0], X, y, svc_hard,   'C = 1e6 (Hard-margin-like)', colors)
plot_svm_boundary(axes[1], X, y, svc_medium, 'C = 1.0 (Soft-margin)', colors)

# Annotate the problem with hard-margin
axes[0].text(0.05, 0.95, f'Many SVs ({svc_hard.n_support_.sum()})\nVery narrow margin\nOverfit risk',
             transform=axes[0].transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

axes[1].text(0.05, 0.95, f'Fewer SVs ({svc_medium.n_support_.sum()})\nWider margin\nBetter generalization',
             transform=axes[1].transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.suptitle('Hard-Margin vs Soft-Margin: The Case for Allowing Mistakes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Section 3: Soft-Margin Dual SVM from Scratch ─────────────────────────────

def soft_margin_dual_svm(X, y, C=1.0, tol=1e-5):
    """
    Solve the soft-margin SVM dual using scipy.optimize.minimize (SLSQP).

    Dual objective (same as hard-margin):
        max  Σ αᵢ − (1/2) Σᵢ Σⱼ αᵢ αⱼ yᵢ yⱼ (xᵢ·xⱼ)

    Constraints:
        0 ≤ αᵢ ≤ C  (BOX constraint — only difference from hard-margin)
        Σᵢ αᵢ yᵢ = 0  (equality constraint)

    Returns: w, b, alphas, sv_mask, xi (slack variables)
    """
    n = len(X)

    # Modified Gram matrix: K[i,j] = yᵢ yⱼ (xᵢ · xⱼ)
    yX = y[:, None] * X
    K  = yX @ yX.T

    # Objective: minimize (1/2) αᵀKα - 1ᵀα  (negated dual max → minimize)
    def objective(alpha):
        return 0.5 * alpha @ K @ alpha - alpha.sum()

    def gradient(alpha):
        return K @ alpha - np.ones(n)

    # Equality constraint: Σᵢ αᵢ yᵢ = 0
    constraints = [{'type': 'eq', 'fun': lambda a: a @ y, 'jac': lambda a: y}]

    # Box bounds: 0 ≤ αᵢ ≤ C  (KEY difference from hard-margin)
    bounds = [(0, C)] * n

    alpha0 = np.zeros(n)
    result = minimize(
        objective, alpha0,
        jac=gradient,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-9, 'maxiter': 2000}
    )

    alphas = np.clip(result.x, 0, C)

    # Recover w: w = Σᵢ αᵢ yᵢ xᵢ
    w = (alphas * y) @ X

    # Recover b using MARGIN support vectors (0 < αᵢ < C → on the margin, ξᵢ=0)
    margin_sv_mask = (alphas > tol) & (alphas < C - tol)
    if margin_sv_mask.sum() > 0:
        b = np.mean(y[margin_sv_mask] - X[margin_sv_mask] @ w)
    else:
        # Fallback: use all SVs
        sv_fallback = alphas > tol
        b = np.mean(y[sv_fallback] - X[sv_fallback] @ w)

    # Support vector mask: any αᵢ > 0
    sv_mask = alphas > tol

    # Slack variables: ξᵢ = max(0, 1 - yᵢ f(xᵢ)) = hinge loss
    f_vals = X @ w + b
    xi = np.maximum(0, 1 - y * f_vals)

    return w, b, alphas, sv_mask, xi


# Run soft-margin dual SVM with C=1
C_test = 1.0
w_soft, b_soft, alphas_soft, sv_mask_soft, xi_soft = soft_margin_dual_svm(X, y, C=C_test)

print(f"Soft-Margin Dual SVM (C={C_test}):")
print(f"  w: [{w_soft[0]:.4f}, {w_soft[1]:.4f}]")
print(f"  b: {b_soft:.4f}")
print(f"  SVs: {sv_mask_soft.sum()} / {len(y)}")
print(f"  Margin SVs (0 < α < C): {((alphas_soft > 1e-5) & (alphas_soft < C_test - 1e-5)).sum()}")
print(f"  Violator SVs (α = C):   {(alphas_soft > C_test - 1e-5).sum()}")
print(f"  Total slack Σξᵢ: {xi_soft.sum():.4f}")
print(f"  Misclassified (ξᵢ > 1): {(xi_soft > 1).sum()}")

In [ ]:
# ─── Section 4: Verify vs sklearn SVC ─────────────────────────────────────────

C_vals_verify = [0.1, 1.0, 10.0]

print("=" * 70)
print("VERIFICATION: Our Dual SVM vs sklearn SVC (linear kernel)")
print("=" * 70)
print(f"{'C':>6} | {'w[0] (ours)':>12} {'w[0] (sk)':>12} | {'w[1] (ours)':>12} {'w[1] (sk)':>12} | {'Match?':>8}")
print("-" * 70)

for C_v in C_vals_verify:
    # Our dual
    w_our, b_our, a_our, sv_our, xi_our = soft_margin_dual_svm(X, y, C=C_v)

    # sklearn
    sk = SVC(kernel='linear', C=C_v)
    sk.fit(X, y)
    w_sk = sk.coef_[0]

    # Normalize both for comparison
    w_our_n = w_our / np.linalg.norm(w_our)
    w_sk_n  = w_sk  / np.linalg.norm(w_sk)
    cos_sim = np.abs(w_our_n @ w_sk_n)

    match = "YES" if cos_sim > 0.999 else f"~{cos_sim:.4f}"
    print(f"{C_v:>6.1f} | {w_our[0]:>12.4f} {w_sk[0]:>12.4f} | {w_our[1]:>12.4f} {w_sk[1]:>12.4f} | {match:>8}")

print("\nNote: Minor numerical differences are expected due to different QP solvers.")
print("Cosine similarity > 0.999 confirms the solutions are effectively identical.")

In [ ]:
# ─── Section 5: C Sweep — Decision Boundary for 6 Values of C ─────────────────

C_values = [0.001, 0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(2, 3, figsize=(17, 11))
axes_flat = axes.flatten()

metrics = []

for ax, C_val in zip(axes_flat, C_values):
    svc = SVC(kernel='linear', C=C_val)
    svc.fit(X, y)

    w   = svc.coef_[0]
    b   = svc.intercept_[0]
    n_sv  = svc.n_support_.sum()
    acc   = accuracy_score(y, svc.predict(X))
    margin = 2 / np.linalg.norm(w)

    # Compute total slack
    f_vals = X @ w + b
    xi_all = np.maximum(0, 1 - y * f_vals)
    total_slack = xi_all.sum()
    n_misclass  = (xi_all > 1).sum()

    metrics.append({
        'C': C_val, 'n_sv': n_sv, 'accuracy': acc,
        'margin': margin, 'total_slack': total_slack, 'n_misclass': n_misclass
    })

    # Plot decision boundary
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 250),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 250)
    )
    Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-10, 0, 10], colors=['#aed6f1', '#fadbd8'], alpha=0.35)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               colors=['#2980b9', '#2c3e50', '#c0392b'],
               linewidths=[1.2, 2.2, 1.2], linestyles=['--', '-', '--'])

    is_spam = (y == 1)
    ax.scatter(X[is_spam, 0],  X[is_spam, 1],  c='#e74c3c', s=30, alpha=0.6, edgecolors='none')
    ax.scatter(X[~is_spam, 0], X[~is_spam, 1], c='#2ecc71', s=30, alpha=0.6, edgecolors='none')

    # Annotate SVs
    sv_idx = svc.support_
    ax.scatter(X[sv_idx, 0], X[sv_idx, 1], s=140, facecolors='none',
               edgecolors='#2c3e50', linewidths=1.8)

    # Color misclassified points in orange
    misclass = xi_all > 1
    if misclass.sum() > 0:
        ax.scatter(X[misclass, 0], X[misclass, 1], s=80, c='#f39c12',
                   marker='x', linewidths=2, zorder=5, label=f'{misclass.sum()} misclass')
        ax.legend(fontsize=7, loc='lower right')

    ax.set_title(
        f'C = {C_val}\n'
        f'SVs={n_sv} | Margin={margin:.3f} | Acc={acc*100:.1f}%\n'
        f'TotalSlack={total_slack:.2f} | Misclass={n_misclass}',
        fontsize=9.5, fontweight='bold'
    )
    ax.set_xlabel('Word Count (Scaled)', fontsize=9)
    ax.set_ylabel('Spam Ratio (Scaled)', fontsize=9)

# Legend patches
red_p   = mpatches.Patch(color='#e74c3c', label='Spam')
grn_p   = mpatches.Patch(color='#2ecc71', label='Ham')
sv_p    = mpatches.Patch(facecolor='none', edgecolor='#2c3e50', label='Support Vector')
mis_p   = mpatches.Patch(color='#f39c12', label='Misclassified')
fig.legend(handles=[red_p, grn_p, sv_p, mis_p], loc='lower center',
           ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Effect of C on Decision Boundary, Margin Width, and Support Vectors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('c_sweep_boundaries.png', dpi=110, bbox_inches='tight')
plt.show()
print("Figure saved as c_sweep_boundaries.png")

metrics_df = pd.DataFrame(metrics)
print("\nMetrics Summary:")
print(metrics_df.to_string(index=False))

In [ ]:
# ─── Section 6: C Sweep Metrics — How C Affects SVs and Margin ───────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

c_log = np.log10(metrics_df['C'])
c_labels = [str(c) for c in metrics_df['C']]

# --- 1. Number of SVs ---
axes[0, 0].semilogx(metrics_df['C'], metrics_df['n_sv'], 'o-', color='#9b59b6', lw=2, ms=8)
axes[0, 0].set_xlabel('C (log scale)', fontsize=11)
axes[0, 0].set_ylabel('Number of Support Vectors', fontsize=11)
axes[0, 0].set_title('C vs Number of SVs\n(High C → fewer SVs)', fontsize=11, fontweight='bold')
for x, y_val, lab in zip(metrics_df['C'], metrics_df['n_sv'], metrics_df['n_sv']):
    axes[0, 0].annotate(str(lab), (x, y_val), textcoords='offset points', xytext=(5, 5), fontsize=9)

# --- 2. Margin Width ---
axes[0, 1].semilogx(metrics_df['C'], metrics_df['margin'], 's-', color='#2980b9', lw=2, ms=8)
axes[0, 1].set_xlabel('C (log scale)', fontsize=11)
axes[0, 1].set_ylabel('Margin Width (2/||w||)', fontsize=11)
axes[0, 1].set_title('C vs Margin Width\n(High C → narrower margin)', fontsize=11, fontweight='bold')
for x, y_val in zip(metrics_df['C'], metrics_df['margin']):
    axes[0, 1].annotate(f'{y_val:.3f}', (x, y_val), textcoords='offset points', xytext=(5, 5), fontsize=9)

# --- 3. Training Accuracy ---
axes[1, 0].semilogx(metrics_df['C'], metrics_df['accuracy'] * 100, '^-', color='#27ae60', lw=2, ms=8)
axes[1, 0].set_xlabel('C (log scale)', fontsize=11)
axes[1, 0].set_ylabel('Training Accuracy (%)', fontsize=11)
axes[1, 0].set_title('C vs Training Accuracy\n(High C → fits training data better)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylim([75, 102])
for x, y_val in zip(metrics_df['C'], metrics_df['accuracy'] * 100):
    axes[1, 0].annotate(f'{y_val:.1f}%', (x, y_val), textcoords='offset points', xytext=(5, -12), fontsize=9)

# --- 4. Total Slack vs Number of Misclassified ---
ax4 = axes[1, 1]
l1, = ax4.semilogx(metrics_df['C'], metrics_df['total_slack'], 'D-', color='#e74c3c', lw=2, ms=8, label='Total Slack Σξᵢ')
ax4b = ax4.twinx()
l2, = ax4b.semilogx(metrics_df['C'], metrics_df['n_misclass'], 'v--', color='#f39c12', lw=2, ms=8, label='# Misclassified')
ax4.set_xlabel('C (log scale)', fontsize=11)
ax4.set_ylabel('Total Slack Σξᵢ', fontsize=11, color='#e74c3c')
ax4b.set_ylabel('# Misclassified', fontsize=11, color='#f39c12')
ax4.set_title('C vs Slack & Misclassifications\n(Low C → more slack allowed)', fontsize=11, fontweight='bold')
ax4.legend(handles=[l1, l2], loc='upper right', fontsize=9)

plt.suptitle('How C Controls the Bias-Variance Tradeoff in Soft-Margin SVM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observations:")
print("  As C increases: fewer SVs, narrower margin, higher training accuracy, less slack")
print("  As C decreases: more SVs, wider margin, lower training accuracy, more slack")
print("  Optimal C balances these tradeoffs (use cross-validation to find it)")

In [ ]:
# ─── Section 7: Slack Variables — Visualize ξᵢ for Each Training Point ────────

C_for_slack = 1.0
svc_c1 = SVC(kernel='linear', C=C_for_slack)
svc_c1.fit(X, y)
w_c1 = svc_c1.coef_[0]
b_c1 = svc_c1.intercept_[0]

# Compute slack: ξᵢ = max(0, 1 - yᵢ f(xᵢ))
f_vals_c1 = X @ w_c1 + b_c1
func_margins_c1 = y * f_vals_c1
xi_c1 = np.maximum(0, 1 - func_margins_c1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: Scatter plot colored by ξᵢ magnitude ---
# Use a diverging colormap: green (ξ=0) → orange (ξ=0.5) → red (ξ>1)
norm  = mcolors.Normalize(vmin=0, vmax=2)
cmap  = plt.cm.RdYlGn_r

scatter = axes[0].scatter(X[:, 0], X[:, 1], c=xi_c1, cmap=cmap, norm=norm,
                           s=70, edgecolors='k', linewidths=0.3, zorder=3)
plt.colorbar(scatter, ax=axes[0], label='Slack ξᵢ = max(0, 1-yᵢf(xᵢ))')

# Draw decision boundary and margin lines
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 250),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 250)
)
Z_c1 = svc_c1.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contour(xx, yy, Z_c1, levels=[-1, 0, 1],
                colors=['#2980b9', '#2c3e50', '#c0392b'],
                linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])

# Label the zones
axes[0].text(0.02, 0.02, 'ξ=0 (correct, outside margin)',
             transform=axes[0].transAxes, fontsize=8, color='green')
axes[0].text(0.02, 0.07, '0<ξ<1 (inside margin, correct)',
             transform=axes[0].transAxes, fontsize=8, color='orange')
axes[0].text(0.02, 0.12, 'ξ>1 (MISCLASSIFIED)',
             transform=axes[0].transAxes, fontsize=8, color='red')

axes[0].set_xlabel('Word Count (Scaled)', fontsize=11)
axes[0].set_ylabel('Spam Ratio (Scaled)', fontsize=11)
axes[0].set_title(f'Slack Variables ξᵢ (C={C_for_slack})\nGreen=no violation, Red=misclassified',
                  fontsize=12, fontweight='bold')

# --- Right: Slack value distribution ---
xi_bins = [
    ('ξ=0\n(no violation)', (xi_c1 == 0).sum(), '#27ae60'),
    ('0<ξ<1\n(margin violator)', ((xi_c1 > 0) & (xi_c1 <= 1)).sum(), '#f39c12'),
    ('ξ>1\n(misclassified)', (xi_c1 > 1).sum(), '#e74c3c')
]
labels_xi  = [b[0] for b in xi_bins]
counts_xi  = [b[1] for b in xi_bins]
colors_xi  = [b[2] for b in xi_bins]

bars = axes[1].bar(labels_xi, counts_xi, color=colors_xi, edgecolor='k', lw=0.5, width=0.5)
for bar, cnt in zip(bars, counts_xi):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(cnt), ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[1].set_ylabel('Number of Training Points', fontsize=11)
axes[1].set_title(f'Slack Variable Categories (C={C_for_slack})\nTotal slack Σξᵢ = {xi_c1.sum():.2f}',
                  fontsize=12, fontweight='bold')
axes[1].set_ylim(0, max(counts_xi) * 1.25)

# Add total slack annotation
axes[1].text(0.5, 0.92, f'Total penalty C·Σξᵢ = {C_for_slack * xi_c1.sum():.2f}',
             transform=axes[1].transAxes, ha='center', fontsize=10,
             bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

plt.suptitle(f'Slack Variables: How Much Does Each Point Violate the Margin? (C={C_for_slack})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Slack summary (C={C_for_slack}):")
print(f"  ξ=0 (correctly classified, outside margin): {(xi_c1 == 0).sum()}")
print(f"  0<ξ≤1 (correctly classified, inside margin): {((xi_c1 > 0) & (xi_c1 <= 1)).sum()}")
print(f"  ξ>1 (misclassified):                         {(xi_c1 > 1).sum()}")
print(f"  Total slack Σξᵢ = {xi_c1.sum():.4f}")
print(f"  Primal objective = (1/2)||w||² + C·Σξᵢ = {0.5*np.dot(w_c1,w_c1):.4f} + {C_for_slack * xi_c1.sum():.4f} = {0.5*np.dot(w_c1,w_c1) + C_for_slack * xi_c1.sum():.4f}")

In [ ]:
# ─── Section 8: Hinge Loss — The Link Between Slack and Loss Function ─────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Left: Hinge loss function ---
yf_vals = np.linspace(-2.5, 3.0, 400)
hinge   = np.maximum(0, 1 - yf_vals)
zero_one = (yf_vals < 0).astype(float)  # 0-1 loss (ideal but non-convex)

ax = axes[0]
ax.plot(yf_vals, hinge,    color='#e74c3c', lw=2.5, label='Hinge loss: max(0, 1−yf(x))')
ax.plot(yf_vals, zero_one, color='#2c3e50', lw=2, linestyle='--', label='0-1 loss (ideal, non-convex)')
ax.axvline(0,  color='gray', lw=1, linestyle=':')
ax.axvline(1,  color='#2980b9', lw=1.5, linestyle='--', label='Margin boundary (yf=1)')
ax.axhline(0,  color='gray', lw=0.8)
ax.fill_between(yf_vals, 0, hinge, where=(hinge > 0), alpha=0.15, color='#e74c3c', label='Penalty region')
ax.set_xlim(-2.5, 3.0)
ax.set_ylim(-0.2, 3.5)
ax.set_xlabel('Functional margin yᵢf(xᵢ)', fontsize=11)
ax.set_ylabel('Loss value', fontsize=11)
ax.set_title('Hinge Loss\n= Slack Variable ξᵢ at Optimum', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Annotate regions
ax.annotate('Misclassified\n(ξ > 1)', xy=(-1, 2), xytext=(-1.8, 2.8),
            arrowprops=dict(arrowstyle='->', lw=1.2), fontsize=8, color='#c0392b')
ax.annotate('In margin\n(0 < ξ < 1)', xy=(0.5, 0.5), xytext=(0.8, 1.5),
            arrowprops=dict(arrowstyle='->', lw=1.2), fontsize=8, color='#e67e22')
ax.annotate('Correct\n(ξ = 0)', xy=(2.0, 0), xytext=(1.8, 0.6),
            arrowprops=dict(arrowstyle='->', lw=1.2), fontsize=8, color='#27ae60')

# --- Middle: Compare several loss functions ---
ax2 = axes[1]
squared_hinge = np.maximum(0, 1 - yf_vals)**2
logistic_loss = np.log(1 + np.exp(-yf_vals))

ax2.plot(yf_vals, hinge,          color='#e74c3c', lw=2.5, label='Hinge (SVM)')
ax2.plot(yf_vals, squared_hinge,  color='#9b59b6', lw=2.0, label='Squared Hinge')
ax2.plot(yf_vals, logistic_loss,  color='#2980b9', lw=2.0, label='Log loss (Logistic Reg)')
ax2.plot(yf_vals, zero_one,       color='#2c3e50', lw=2.0, linestyle='--', label='0-1 loss (ideal)')
ax2.axvline(1, color='gray', lw=1, linestyle='--')
ax2.set_xlim(-2.5, 3.0)
ax2.set_ylim(-0.2, 4.0)
ax2.set_xlabel('Functional margin yᵢf(xᵢ)', fontsize=11)
ax2.set_ylabel('Loss value', fontsize=11)
ax2.set_title('Loss Functions Comparison', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

# --- Right: Distribution of functional margins for our C=1 model ---
ax3 = axes[2]
is_spam = (y == 1)

ax3.hist(func_margins_c1[is_spam],  bins=20, alpha=0.6, color='#e74c3c',
         label='Spam (y=+1)', edgecolor='k', lw=0.3)
ax3.hist(func_margins_c1[~is_spam], bins=20, alpha=0.6, color='#2ecc71',
         label='Ham (y=-1)', edgecolor='k', lw=0.3)
ax3.axvline(-1, color='#2980b9', lw=2, linestyle='--', label='Margin −1')
ax3.axvline( 0, color='k',       lw=2, label='Decision boundary')
ax3.axvline( 1, color='#c0392b', lw=2, linestyle='--', label='Margin +1')

# Shade the hinge loss region (|margin| < 1 = violation)
ymin_ax, ymax_ax = ax3.get_ylim()
ax3.axvspan(-1, 1, alpha=0.08, color='purple', label='Margin zone (ξ>0 here)')
ax3.set_xlabel('Functional margin yᵢf(xᵢ)', fontsize=11)
ax3.set_ylabel('Count', fontsize=11)
ax3.set_title(f'Functional Margin Distribution (C={C_for_slack})\nPoints in purple zone have ξ > 0',
              fontsize=11, fontweight='bold')
ax3.legend(fontsize=8, loc='upper left')

plt.suptitle('Hinge Loss = Slack Variables: SVM as Regularized ERM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key equivalence:")
print("  Soft-margin SVM objective: (1/2)||w||² + C Σ max(0, 1-yᵢf(xᵢ))")
print("  = L2 regularizer          + C × hinge loss")
print("  = (1/λ) × hinge loss      + ||w||²    where λ = 1/C")
print("  SVM is ridge regression with hinge loss instead of squared error!")

In [ ]:
# ─── Detailed SV Type Analysis — Margin SVs vs Violator SVs ──────────────────

C_analysis = 1.0
svc_an = SVC(kernel='linear', C=C_analysis)
svc_an.fit(X, y)
w_an = svc_an.coef_[0]
b_an = svc_an.intercept_[0]

f_an       = X @ w_an + b_an
func_m_an  = y * f_an
xi_an      = np.maximum(0, 1 - func_m_an)

# Use dual to get alphas for analysis
w_d, b_d, alphas_d, sv_d, xi_d = soft_margin_dual_svm(X, y, C=C_analysis)

# Categorize each training point
categories = []
for i in range(len(y)):
    if alphas_d[i] < 1e-5:
        categories.append('Non-SV (α=0)')
    elif alphas_d[i] > C_analysis - 1e-5:
        if xi_d[i] > 1:
            categories.append('Misclassified SV (α=C, ξ>1)')
        else:
            categories.append('Margin Violator (α=C, 0<ξ≤1)')
    else:
        categories.append('Margin SV (0<α<C, ξ=0)')

cat_counts = pd.Series(categories).value_counts()
print("Support Vector Type Analysis (C=1.0):")
print("=" * 55)
print(cat_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: Scatter with SV types annotated ---
cat_colors = {
    'Non-SV (α=0)':                  '#bdc3c7',
    'Margin SV (0<α<C, ξ=0)':        '#2980b9',
    'Margin Violator (α=C, 0<ξ≤1)':  '#f39c12',
    'Misclassified SV (α=C, ξ>1)':   '#e74c3c'
}
cat_markers = {
    'Non-SV (α=0)': 'o',
    'Margin SV (0<α<C, ξ=0)': 's',
    'Margin Violator (α=C, 0<ξ≤1)': 'D',
    'Misclassified SV (α=C, ξ>1)': 'X'
}

for cat in cat_colors:
    mask_cat = np.array(categories) == cat
    if mask_cat.sum() > 0:
        axes[0].scatter(X[mask_cat, 0], X[mask_cat, 1],
                        c=cat_colors[cat], marker=cat_markers[cat],
                        s=70, edgecolors='k', linewidths=0.5, alpha=0.85,
                        label=f'{cat} ({mask_cat.sum()})', zorder=4)

# Draw decision boundary
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 250),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 250)
)
Z_an = svc_an.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contour(xx, yy, Z_an, levels=[-1, 0, 1],
                colors=['#2980b9', '#2c3e50', '#c0392b'],
                linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])

axes[0].set_xlabel('Word Count (Scaled)', fontsize=11)
axes[0].set_ylabel('Spam Ratio (Scaled)', fontsize=11)
axes[0].set_title(f'SV Types: C={C_analysis}\nDashed lines = margin boundaries',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='upper left')

# --- Right: Bar chart of α values colored by type ---
cat_array = np.array(categories)
bar_c2    = [cat_colors[c] for c in categories]
sorted_by_alpha = np.argsort(alphas_d)[::-1]
bar_c2_sorted   = [bar_c2[i] for i in sorted_by_alpha]

axes[1].bar(range(len(alphas_d)), alphas_d[sorted_by_alpha],
            color=bar_c2_sorted, edgecolor='none')
axes[1].axhline(C_analysis, color='k', linestyle='--', lw=1.5, label=f'C = {C_analysis}')
axes[1].axhline(0,          color='gray', lw=0.8)
axes[1].set_xlabel('Training Point (sorted by αᵢ)', fontsize=11)
axes[1].set_ylabel('Lagrange Multiplier αᵢ', fontsize=11)
axes[1].set_title('α Values — Three Regimes\n(αᵢ=0, 0<αᵢ<C, αᵢ=C)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

# Add region labels
n_at_c = (alphas_d > C_analysis - 1e-5).sum()
n_between = ((alphas_d > 1e-5) & (alphas_d < C_analysis - 1e-5)).sum()
axes[1].text(n_at_c/2, C_analysis * 1.05, f'αᵢ=C: {n_at_c} violators',
             ha='center', fontsize=9, color='#e74c3c')
axes[1].text(n_at_c + n_between/2, C_analysis * 0.5,
             f'0<αᵢ<C:\n{n_between} margin SVs',
             ha='center', fontsize=9, color='#2980b9')

plt.suptitle(f'Three Regimes of Support Vectors in Soft-Margin SVM (C={C_analysis})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── C vs Regularization: λ = 1/C Interpretation ─────────────────────────────

print("The SVM objective with regularization notation:")
print()
print("  Soft-margin: (1/2)||w||² + C Σ max(0, 1-yᵢf(xᵢ))")
print("  Divide by C: (1/2C)||w||² + Σ max(0, 1-yᵢf(xᵢ))")
print("  Let λ = 1/C: λ||w||² + Σ hinge_loss")
print()
print("  → Large C  = small λ = weak regularization = fits data tightly")
print("  → Small C  = large λ = strong regularization = wider margin, ignores outliers")
print("  → This is identical to L2-regularized ERM with hinge loss!")
print()

# Demonstration: compute primal objective for each C
primal_objectives = []
for C_v in C_values:
    svc_tmp = SVC(kernel='linear', C=C_v)
    svc_tmp.fit(X, y)
    w_tmp = svc_tmp.coef_[0]
    b_tmp = svc_tmp.intercept_[0]
    f_tmp = X @ w_tmp + b_tmp
    xi_tmp = np.maximum(0, 1 - y * f_tmp)
    reg_term   = 0.5 * np.dot(w_tmp, w_tmp)
    hinge_term = C_v * xi_tmp.sum()
    primal_objectives.append({
        'C': C_v, 'lambda': 1/C_v,
        'reg_term': reg_term, 'hinge_term': hinge_term,
        'total': reg_term + hinge_term
    })

obj_df = pd.DataFrame(primal_objectives)
print("Primal objective breakdown by C:")
print(obj_df.to_string(index=False, float_format='{:.4f}'.format))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x_pos  = np.arange(len(C_values))
width  = 0.35
bars1 = ax.bar(x_pos - width/2, obj_df['reg_term'],  width, label='(1/2)||w||² (margin term)',  color='#2980b9', alpha=0.85, edgecolor='k', lw=0.4)
bars2 = ax.bar(x_pos + width/2, obj_df['hinge_term'], width, label='C × Σhinge (penalty term)', color='#e74c3c', alpha=0.85, edgecolor='k', lw=0.4)
ax.plot(x_pos, obj_df['total'], 'k^-', lw=2, ms=8, label='Total primal objective')

ax.set_xticks(x_pos)
ax.set_xticklabels([f'C={c}\n(λ={1/c:.3f})' for c in C_values], fontsize=9)
ax.set_ylabel('Objective value', fontsize=11)
ax.set_title('Primal Objective Decomposition\n(Regularization term vs Hinge loss penalty)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Full Summary Visualization ───────────────────────────────────────────────

fig = plt.figure(figsize=(17, 11))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

is_spam = (y == 1)

# 1. Overlapping dataset
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(X[is_spam, 0],  X[is_spam, 1],  c='#e74c3c', s=35, alpha=0.65, edgecolors='none', label='Spam')
ax1.scatter(X[~is_spam, 0], X[~is_spam, 1], c='#2ecc71', s=35, alpha=0.65, edgecolors='none', label='Ham')
ax1.axvspan(-0.8, 0.8, alpha=0.08, color='purple', label='Overlap')
ax1.set_title('Non-Separable Data', fontweight='bold', fontsize=11)
ax1.legend(fontsize=8)

# 2. C=0.01 (wide margin, many errors)
ax2 = fig.add_subplot(gs[0, 1])
svc01 = SVC(kernel='linear', C=0.01).fit(X, y)
Z01 = svc01.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z01, levels=[-10,0,10], colors=['#aed6f1','#fadbd8'], alpha=0.35)
ax2.contour(xx, yy, Z01, levels=[-1,0,1], colors=['#2980b9','k','#c0392b'], linewidths=[1.2,2,1.2])
ax2.scatter(X[is_spam,0], X[is_spam,1],   c='#e74c3c', s=25, alpha=0.6, edgecolors='none')
ax2.scatter(X[~is_spam,0], X[~is_spam,1], c='#2ecc71', s=25, alpha=0.6, edgecolors='none')
xi01 = np.maximum(0, 1 - y * (X @ svc01.coef_[0] + svc01.intercept_[0]))
m01  = 2 / np.linalg.norm(svc01.coef_[0])
ax2.set_title(f'C=0.01: Wide Margin\nMargin={m01:.3f}, SVs={svc01.n_support_.sum()}', fontweight='bold', fontsize=11)

# 3. C=10 (narrow margin, fewer errors)
ax3 = fig.add_subplot(gs[0, 2])
svc10 = SVC(kernel='linear', C=10).fit(X, y)
Z10 = svc10.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax3.contourf(xx, yy, Z10, levels=[-10,0,10], colors=['#aed6f1','#fadbd8'], alpha=0.35)
ax3.contour(xx, yy, Z10, levels=[-1,0,1], colors=['#2980b9','k','#c0392b'], linewidths=[1.2,2,1.2])
ax3.scatter(X[is_spam,0], X[is_spam,1],   c='#e74c3c', s=25, alpha=0.6, edgecolors='none')
ax3.scatter(X[~is_spam,0], X[~is_spam,1], c='#2ecc71', s=25, alpha=0.6, edgecolors='none')
m10 = 2 / np.linalg.norm(svc10.coef_[0])
ax3.set_title(f'C=10: Narrow Margin\nMargin={m10:.3f}, SVs={svc10.n_support_.sum()}', fontweight='bold', fontsize=11)

# 4. Slack distribution for C=1
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(xi_c1[xi_c1 == 0],                        bins=1,  alpha=0.8, color='#27ae60', label=f'ξ=0: {(xi_c1==0).sum()}')
ax4.hist(xi_c1[(xi_c1>0)&(xi_c1<=1)],              bins=10, alpha=0.8, color='#f39c12', label=f'0<ξ≤1: {((xi_c1>0)&(xi_c1<=1)).sum()}')
ax4.hist(xi_c1[xi_c1>1],                           bins=5,  alpha=0.8, color='#e74c3c', label=f'ξ>1: {(xi_c1>1).sum()}')
ax4.set_xlabel('Slack ξᵢ', fontsize=10)
ax4.set_ylabel('Count', fontsize=10)
ax4.set_title('Slack Distribution (C=1)', fontweight='bold', fontsize=11)
ax4.legend(fontsize=8)

# 5. C vs SVs and margin
ax5 = fig.add_subplot(gs[1, 1])
ax5.semilogx(metrics_df['C'], metrics_df['n_sv'],  'o-', color='#9b59b6', lw=2, ms=7, label='# SVs')
ax5b = ax5.twinx()
ax5b.semilogx(metrics_df['C'], metrics_df['margin'], 's--', color='#2980b9', lw=2, ms=7, label='Margin width')
ax5.set_xlabel('C', fontsize=10)
ax5.set_ylabel('# Support Vectors', fontsize=10, color='#9b59b6')
ax5b.set_ylabel('Margin Width', fontsize=10, color='#2980b9')
ax5.set_title('C vs SVs and Margin\n(Opposite trends)', fontweight='bold', fontsize=11)
lines1, labels1 = ax5.get_legend_handles_labels()
lines2, labels2 = ax5b.get_legend_handles_labels()
ax5.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# 6. Hinge loss
ax6 = fig.add_subplot(gs[1, 2])
yf_range = np.linspace(-2, 3, 300)
ax6.plot(yf_range, np.maximum(0, 1 - yf_range), color='#e74c3c', lw=2.5, label='Hinge loss')
ax6.fill_between(yf_range, 0, np.maximum(0, 1-yf_range), where=(yf_range < 1), alpha=0.2, color='#e74c3c')
ax6.axvline(0, color='k', lw=1.5, linestyle='--', label='Decision boundary')
ax6.axvline(1, color='#2980b9', lw=1.5, linestyle='--', label='Margin')
ax6.set_xlabel('yᵢf(xᵢ)', fontsize=10)
ax6.set_ylabel('Hinge Loss = ξᵢ', fontsize=10)
ax6.set_title('Hinge Loss = Slack Variable\n(Penalizes margin violations)', fontweight='bold', fontsize=11)
ax6.legend(fontsize=8)
ax6.set_ylim(-0.1, 3)

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('Word Count (Scaled)', fontsize=9)
    ax.set_ylabel('Spam Ratio (Scaled)', fontsize=9)

fig.suptitle('Soft-Margin SVM — Complete Summary', fontsize=15, fontweight='bold')
plt.savefig('soft_margin_svm_summary.png', dpi=110, bbox_inches='tight')
plt.show()
print("Summary figure saved as soft_margin_svm_summary.png")

## Key Formulas Reference

| Symbol | Meaning | Range |
|---|---|---|
| $\xi_i$ | Slack variable (degree of margin violation) | $[0, \infty)$ |
| $C$ | Penalty for violations (hyperparameter) | $(0, \infty)$ |
| $\alpha_i$ | Lagrange multiplier for point $i$ | $[0, C]$ |
| $2/\|w\|$ | Margin width | $(0, \infty)$ |

**Primal:**
$$\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 - \xi_i, \; \xi_i \geq 0$$

**Dual:**
$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j (x_i \cdot x_j) \quad \text{s.t.} \quad 0 \leq \alpha_i \leq C, \; \sum_i \alpha_i y_i = 0$$

**Hinge loss (= slack at optimum):**
$$\xi_i = \max(0, 1 - y_i f(x_i))$$

**Equivalent formulation:**
$$\text{SVM} = \text{L2 regularization} + C \times \text{hinge loss} = \lambda \|w\|^2 + \sum_i \max(0, 1 - y_i f(x_i))$$

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** As C → ∞, the soft-margin SVM approaches the hard-margin SVM. Why does increasing C make the margin narrower?

<details>
<summary>Click to reveal answer</summary>

The objective is $\frac{1}{2}\|w\|^2 + C \sum \xi_i$. When C is very large, the penalty term dominates:

- The optimizer is forced to make $\xi_i$ as small as possible (ideally 0), because each unit of slack costs C (a huge amount)
- To drive all $\xi_i \to 0$, the decision boundary must correctly classify every point with margin ≥ 1
- This forces $\|w\|$ to increase (the margin $2/\|w\|$ narrows) to correctly classify hard points

Intuitively: a very strict penalty means no violations are tolerated. The SVM squeezes the margin to accommodate every training point, even outliers — which is exactly the hard-margin behavior.

Formally: when $C \to \infty$, the constraint $\xi_i \geq 0$ forces $\xi_i = 0$, and the constraint $y_i(w \cdot x_i + b) \geq 1 - \xi_i$ becomes $y_i(w \cdot x_i + b) \geq 1$ — the hard-margin constraint.

</details>

---

**Question 2:** Point A has ξ=0.4 (inside margin, correct side). Point B has ξ=1.5 (misclassified). Which incurs a higher penalty? Which is a support vector? What is each point's Lagrange multiplier αᵢ?

<details>
<summary>Click to reveal answer</summary>

**Penalty:**
- Point A: penalty = C × 0.4
- Point B: penalty = C × 1.5
- Point B incurs a higher penalty (3.75× higher than A)

**Both are support vectors!** A support vector in the soft-margin SVM is any point with $\alpha_i > 0$, which means any point that violates or is exactly on the margin:
- Point A ($\xi = 0.4 > 0$): it's inside the margin → $\alpha_i = C$ (a violator SV)
- Point B ($\xi = 1.5 > 0$): misclassified → $\alpha_i = C$ (a violator SV, even worse)

**Lagrange multipliers:**
- Both have $\alpha_i = C$ (they've hit the upper bound of the box constraint)
- Only points exactly ON the margin (ξ=0, yᵢf(xᵢ)=1) have $0 < \alpha_i < C$
- Points comfortably outside the margin (ξ=0, yᵢf(xᵢ)>1) have $\alpha_i = 0$ (non-SVs)

The KKT condition for soft-margin: $\alpha_i [y_i f(x_i) - 1 + \xi_i] = 0$. For violators, $\xi_i > 0$, so $y_i f(x_i) = 1 - \xi_i < 1$, which forces $\alpha_i = C$ to satisfy the condition.

</details>

---

**Question 3:** The soft-margin objective is (1/2)||w||² + C Σ ξᵢ. This is equivalent to regularized hinge loss. What does C correspond to in the regularized loss framework (hint: think λ = 1/C)? What happens to the bias-variance tradeoff as C increases?

<details>
<summary>Click to reveal answer</summary>

Dividing the SVM objective by C:
$$\frac{1}{2C}\|w\|^2 + \sum_i \xi_i \equiv \lambda\|w\|^2 + \sum_i \text{hinge\_loss}(y_i, f(x_i))$$
where $\lambda = \frac{1}{2C}$.

**In regularized ERM notation:** $C$ corresponds to $\frac{1}{\lambda}$ — the **inverse of the regularization strength**.

- Large C → small λ → **weak regularization** → model fits the training data closely → high variance, low bias → risk of overfitting
- Small C → large λ → **strong regularization** → model ignores training details → low variance, high bias → risk of underfitting

**Bias-variance tradeoff:**
- As C increases: training accuracy goes up (fits data better), but the model becomes more sensitive to individual training points → high variance, may overfit on test data
- As C decreases: training accuracy goes down, but the model generalizes by finding a wide margin → low variance, may underfit

This is identical to the regularization tradeoff in ridge regression, lasso, or any other regularized model — SVM is just regularized ERM with hinge loss.

</details>